# Gemma Cross-Dataset Boundary Validation

Train local-evidence MGT classifiers from existing Gemma 3 1B Instruct outputs and validate cross-dataset classification and partial SemEval boundaries.

In [ ]:
from __future__ import annotations

import os
from pathlib import Path

import pandas as pd
from IPython.display import display
from sklearn.metrics import f1_score
from tqdm.auto import tqdm

from notebooks.local_evidence_models_utils import (
    LOCAL_FEATURE_COLUMNS,
    fit_local_evidence_classifier,
    save_fitted_models,
    score_local_evidence_classifier,
    training_input_fingerprint,
)
from notebooks.local_evidence_utils import load_or_build_local_evidence_windows, split_local_evidence_texts
from notebooks.semeval_boundary_utils import (
    add_gold_boundaries,
    predict_boundary_positions,
    summarize_boundary_predictions,
)
from notebooks.token_candidate_similarity_heatmaps_utils import load_token_labels

DATA_ROOT = Path(os.environ["TEXT_DETECTION_DATA"])
PROCESSED_ROOT = DATA_ROOT / "processed"
MODEL = "gemma_3_1b_it"
SOURCE_DATASETS = ("kaggle_train", "mage_train", "outfox_chatgpt_train")
SEMEVAL_DATASET = "semeval2024_task8_subtask_c_dev"
SEMEVAL_SPLIT = "MG"
SEMEVAL_TEXT_IDS = tuple(range(270))
SEMEVAL_TOTAL_TEXTS = 505
WINDOW_LENGTHS = (32, 64, 128)
STRIDE = 16
TOP_K = 20
SEED = 42
THRESHOLD = 0.5
SMOOTHING_WINDOW = 5
SUSTAINED_LENGTH = 3
MODEL_CACHE_ROOT = PROCESSED_ROOT / "gemma_3_1b_it_local_evidence_models"
LOCAL_EVIDENCE_CACHE_ROOT = PROCESSED_ROOT / "gemma_3_1b_it_local_evidence_windows"
WINDOW_COLUMNS = ("split", "label", "text_id", "window_length", "start_token", "stop_token", *LOCAL_FEATURE_COLUMNS)

def load_compact_windows(dataset: str, splits: tuple[str, ...]) -> pd.DataFrame:
    windows = load_or_build_local_evidence_windows(
        PROCESSED_ROOT, dataset, MODEL, path=LOCAL_EVIDENCE_CACHE_ROOT / f"{dataset}.parquet",
        window_lengths=WINDOW_LENGTHS, stride=STRIDE, top_k=TOP_K, splits=splits, recompute=False,
    ).drop(columns="window_text")
    compact = windows.loc[:, WINDOW_COLUMNS].copy()
    compact.loc[:, list(LOCAL_FEATURE_COLUMNS)] = compact.loc[:, list(LOCAL_FEATURE_COLUMNS)].astype("float32")
    return compact


In [ ]:
source_windows = {}
coverage_rows = []
for dataset in tqdm(SOURCE_DATASETS, desc="Loading source datasets", unit="dataset"):
    try:
        windows = load_compact_windows(dataset, ("HG", "MG"))
    except (FileNotFoundError, KeyError, ValueError) as error:
        coverage_rows.append({"dataset": dataset, "status": "excluded", "reason": str(error)})
        continue
    source_windows[dataset] = windows
    for split, group in windows.groupby("split", sort=True):
        coverage_rows.append({
            "dataset": dataset, "status": "available", "split": split,
            "texts": group["text_id"].nunique(), "windows": len(group), "reason": "",
        })
coverage = pd.DataFrame(coverage_rows)
display(coverage)

semeval_windows = load_compact_windows(SEMEVAL_DATASET, (SEMEVAL_SPLIT,))
semeval_windows = semeval_windows.loc[semeval_windows["text_id"].isin(SEMEVAL_TEXT_IDS)].copy()
semeval_coverage = pd.DataFrame([{
    "model": MODEL, "processed_texts": semeval_windows["text_id"].nunique(),
    "total_texts": SEMEVAL_TOTAL_TEXTS, "coverage_label": "270/505 partial coverage",
}])
display(semeval_coverage)


In [ ]:
source_partitions = {
    dataset: split_local_evidence_texts(windows, seed=SEED)
    for dataset, windows in source_windows.items()
}
regimes = {dataset: source_windows[dataset] for dataset in source_windows}
mixed_windows = pd.concat(
    [frame.assign(split=lambda value, dataset=dataset: dataset + "__" + value["split"])
     for dataset, frame in source_windows.items()],
    ignore_index=True,
)
regimes["mixed"] = mixed_windows
mixed_partitions = split_local_evidence_texts(mixed_windows, seed=SEED)

fitted_models = {}
cross_dataset_rows = []
for regime, training_windows in regimes.items():
    partitions = mixed_partitions if regime == "mixed" else source_partitions[regime]
    train_parents = partitions["train"]
    train_windows = training_windows.merge(
        train_parents[["split", "text_id", "label"]], on=["split", "text_id", "label"]
    )
    test_parents = partitions["test"]
    regime_test_windows = training_windows.merge(
        test_parents[["split", "text_id", "label"]], on=["split", "text_id", "label"]
    )
    regime_fitted = {}
    for window_length in WINDOW_LENGTHS:
        fitted = fit_local_evidence_classifier(
            "logistic_regression", train_windows.query("window_length == @window_length"),
            LOCAL_FEATURE_COLUMNS, SEED,
        )
        fitted_models[(regime, window_length)] = fitted
        regime_fitted[("logistic_regression", window_length)] = fitted
        for target, target_windows in source_windows.items():
            if regime == "mixed":
                evaluation_windows = regime_test_windows.loc[
                    regime_test_windows["split"].str.startswith(target + "__")
                ]
            elif target == regime:
                evaluation_windows = regime_test_windows
            else:
                evaluation_windows = target_windows
            scored = score_local_evidence_classifier(
                fitted, evaluation_windows.query("window_length == @window_length"), LOCAL_FEATURE_COLUMNS,
            )
            text_scores = scored.groupby(["split", "text_id", "label"], as_index=False)["mg_score"].max()
            predictions = (text_scores["mg_score"] >= THRESHOLD).astype("int64")
            cross_dataset_rows.append({
                "regime": regime, "target_dataset": target, "window_length": window_length,
                "texts": len(text_scores),
                "macro_f1": f1_score(text_scores["label"], predictions, average="macro"),
            })
    manifest = {
        "regime": regime, "model": MODEL, "features": list(LOCAL_FEATURE_COLUMNS), "seed": SEED,
        "window_lengths": list(WINDOW_LENGTHS), "model_names": ["logistic_regression"],
        "training_input_fingerprint": training_input_fingerprint(training_windows, partitions),
    }
    save_fitted_models(MODEL_CACHE_ROOT / regime, regime_fitted, manifest)

cross_dataset_summary = pd.DataFrame(cross_dataset_rows)
cross_dataset_matrix = cross_dataset_summary.pivot(
    index=["regime", "window_length"], columns="target_dataset", values="macro_f1"
)
display(cross_dataset_summary.sort_values(["regime", "target_dataset", "window_length"]))
display(cross_dataset_matrix)


In [ ]:
annotations = pd.read_csv(
    DATA_ROOT / "raw/semevals_2024-task8/SemEval2024-task8/subtaskC/SemEval2024_task8_C_dev_MG.csv"
)
token_labels = load_token_labels(PROCESSED_ROOT, SEMEVAL_DATASET, MODEL, SEMEVAL_SPLIT, show_progress=False)
token_counts = {text_id: len(token_labels[text_id]) for text_id in semeval_windows["text_id"].unique()}
boundary_frames = []
exclusion_rows = []
for (regime, window_length), fitted in fitted_models.items():
    scored = score_local_evidence_classifier(
        fitted, semeval_windows.query("window_length == @window_length"), LOCAL_FEATURE_COLUMNS,
    )
    scored["model"] = regime
    predicted = predict_boundary_positions(
        scored, token_counts, threshold=THRESHOLD, smoothing_window=SMOOTHING_WINDOW,
        sustained_length=SUSTAINED_LENGTH, show_progress=False,
    )
    for text_id, frame in predicted.groupby("text_id", sort=True):
        try:
            boundary_frames.append(add_gold_boundaries(frame, annotations, token_labels))
        except ValueError as error:
            exclusion_rows.append({
                "regime": regime, "window_length": window_length, "text_id": int(text_id),
                "reason": str(error),
            })

semeval_predictions = pd.concat(boundary_frames, ignore_index=True)
semeval_exclusions = pd.DataFrame(
    exclusion_rows, columns=["regime", "window_length", "text_id", "reason"]
)
semeval_boundary_summary = summarize_boundary_predictions(semeval_predictions).assign(
    coverage_label="270/505 partial coverage"
)
display(semeval_boundary_summary.sort_values(["model", "window_length"])[[
    "model", "window_length", "texts", "mean_absolute_token_error",
    "exact_accuracy", "within_5_accuracy", "coverage_label",
]])
display(semeval_exclusions.sort_values(["regime", "window_length", "text_id"]))
